<a href="https://colab.research.google.com/github/ayeshakauser07/Neural-Receptor-Ion-Channel-Property-Profiler/blob/main/Neural_Receptor_%26_Ion_Channel_Property_Profiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Receptor & Ion Channel Property Profiler

**Pipeline:** fetch sequence → hydrophobicity scan → charge scan → flag transmembrane regions → plot → summarize → validate against UniProt ground truth

Run each cell top to bottom.

In [ ]:
!pip install biopython -q

## Setup — imports & target proteins

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
from Bio.SeqUtils.ProtParamData import kd  # Kyte-Doolittle hydrophobicity scale

# Choose target proteins
# UniProt accession IDs for well-characterized neural membrane proteins.
# (Accessions are stable identifiers -- safer to hardcode than gene-name search.)
TARGET_PROTEINS = {
    "GRIN1_NMDA_receptor":   "Q05586",
    "GABRA1_GABA-A_receptor": "P14867",
    "SCN1A_sodium_channel":   "P35498",
    "GRIA1_AMPA_receptor":    "P42261",
}


## Step 2 — Get the sequences (fetch FASTA from UniProt REST API)

In [ ]:
def fetch_uniprot_sequence(accession: str) -> str:
    """
    Download a protein's canonical amino acid sequence from UniProt.
    Returns the sequence as a plain string (no header, no line breaks).
    """
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    lines = resp.text.strip().split("\n")
    sequence = "".join(lines[1:])  # skip the ">header" line
    return sequence

## Step 3 — Compute hydrophobicity (sliding window, Kyte-Doolittle scale)

In [ ]:
def sliding_hydrophobicity(sequence: str, window: int = 19) -> np.ndarray:
    """
    Returns an array of average Kyte-Doolittle hydrophobicity per window,
    centered at each residue position. Output length always equals
    len(sequence) (edge-padded), so it aligns exactly with sliding_charge()
    and with real residue positions -- important for correctly locating
    transmembrane regions later.
    """
    raw_scores = np.array([kd.get(aa, 0.0) for aa in sequence])
    half = window // 2
    padded = np.pad(raw_scores, (half, half), mode="edge")
    smoothed = np.array([
        padded[i:i + window].mean() for i in range(len(raw_scores))
    ])
    return smoothed

## Step 4 — Compute net charge (sliding window)

In [ ]:
CHARGE_TABLE = {
    "D": -1, "E": -1,          # negatively charged
    "K": 1, "R": 1,            # positively charged
    "H": 0.1,                  # weakly positive at physiological pH
}

def sliding_charge(sequence: str, window: int = 9) -> np.ndarray:
    """
    Returns an array of average net charge per window across the sequence.
    Window is smaller than the hydrophobicity window since charge clusters
    tend to be sharper/more localized.
    """
    charges = np.array([CHARGE_TABLE.get(aa, 0) for aa in sequence])
    half = window // 2
    padded = np.pad(charges, (half, half), mode="edge")
    smoothed = np.array([
        padded[i:i + window].mean() for i in range(len(charges))
    ])
    return smoothed

## Step 5 — Flag transmembrane candidate regions

Uses **hysteresis thresholding** (same idea as Canny edge detection): a single fixed cutoff only catches the narrow hydrophobic *core* of a real TM helix, since many neural ion-channel/receptor helices have polar, pore-facing residues pulling the average down across much of their length. Fix: find confident high-threshold seeds, then expand outward at a lower threshold to recover the helix's true width, merge nearby fragments, and drop anything still too short to be real.

In [ ]:
def flag_tm_regions(hydro_scores: np.ndarray, high_threshold: float = 1.6,
                     low_threshold: float = 0.5, min_length: int = 15,
                     max_gap: int = 4) -> list:
    """
    Hysteresis thresholding (same principle as Canny edge detection):
    a single fixed threshold only catches the narrow hydrophobic *core*
    of a real TM helix -- many neural ion-channel/receptor TM helices
    have polar, pore-facing residues that pull the average well below
    a strict cutoff across much of the helix, even though the helix is
    still genuinely membrane-spanning. Fix:

      1. Find confident seed positions where hydro >= high_threshold.
      2. Expand each seed left/right while hydro stays >= low_threshold,
         to recover the true full width of the helix.
      3. Merge expanded regions that end up overlapping or within
         max_gap residues of each other.
      4. Discard anything still shorter than min_length (real TM
         helices span ~18-25 residues; short survivors are noise).
    """
    n = len(hydro_scores)

    is_seed = hydro_scores >= high_threshold
    seeds = []
    in_seed = False
    for i, flag in enumerate(is_seed):
        if flag and not in_seed:
            in_seed = True
            start = i
        elif not flag and in_seed:
            in_seed = False
            seeds.append((start, i - 1))
    if in_seed:
        seeds.append((start, n - 1))

    if not seeds:
        return []

    expanded = []
    for s, e in seeds:
        left = s
        while left > 0 and hydro_scores[left - 1] >= low_threshold:
            left -= 1
        right = e
        while right < n - 1 and hydro_scores[right + 1] >= low_threshold:
            right += 1
        expanded.append((left, right))

    expanded.sort()
    merged = [expanded[0]]
    for start, end in expanded[1:]:
        prev_start, prev_end = merged[-1]
        if start - prev_end <= max_gap:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))

    filtered = [(s, e) for s, e in merged if (e - s + 1) >= min_length]

    return filtered

## Step 6 — Plot the profile

In [ ]:
def plot_profile(name: str, hydro: np.ndarray, charge: np.ndarray,
                  tm_regions: list, save_path: str = None):
    """
    Plots hydrophobicity + charge traces along the sequence, with
    shaded bands over flagged transmembrane regions.
    """
    fig, ax1 = plt.subplots(figsize=(11, 4))

    ax1.plot(hydro, color="#c0392b", label="Hydrophobicity (Kyte-Doolittle)")
    ax1.axhline(1.6, color="#c0392b", linestyle="--", linewidth=0.7, alpha=0.5)
    ax1.set_xlabel("Residue position")
    ax1.set_ylabel("Hydrophobicity", color="#c0392b")
    ax1.tick_params(axis="y", labelcolor="#c0392b")

    ax2 = ax1.twinx()
    ax2.plot(charge, color="#2980b9", alpha=0.7, label="Net charge")
    ax2.set_ylabel("Net charge", color="#2980b9")
    ax2.tick_params(axis="y", labelcolor="#2980b9")

    for (start, end) in tm_regions:
        ax1.axvspan(start, end, color="orange", alpha=0.2)

    plt.title(f"Hydrophobicity & Charge Profile — {name}")
    fig.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close()
    else:
        plt.show()

## Step 7 — Summarize output

In [ ]:
def summarize(name: str, tm_regions: list) -> dict:
    """
    Builds a summary record for one protein: number of predicted TM
    domains and their residue ranges.
    """
    return {
        "protein": name,
        "num_predicted_tm_domains": len(tm_regions),
        "regions": tm_regions,
    }

## Step 8 — Run the full pipeline across all target proteins

In [ ]:
def run_pipeline(protein_dict: dict, output_dir: str = "."):
    results = []
    for name, accession in protein_dict.items():
        print(f"Processing {name} ({accession})...")
        seq = fetch_uniprot_sequence(accession)

        hydro = sliding_hydrophobicity(seq)
        charge = sliding_charge(seq)
        tm_regions = flag_tm_regions(hydro)

        plot_profile(name, hydro, charge, tm_regions,
                     save_path=f"{output_dir}/{name}_profile.png")

        summary = summarize(name, tm_regions)
        summary["sequence_length"] = len(seq)
        results.append(summary)

    return results

In [ ]:
all_results = run_pipeline(TARGET_PROTEINS)

print("\n--- Summary Table ---")
for r in all_results:
    print(f"{r['protein']:25s} | length={r['sequence_length']:5d} "
          f"| predicted TM domains={r['num_predicted_tm_domains']}")

Processing GRIN1_NMDA_receptor (Q05586)...
Processing GABRA1_GABA-A_receptor (P14867)...
Processing SCN1A_sodium_channel (P35498)...
Processing GRIA1_AMPA_receptor (P42261)...

--- Summary Table ---
GRIN1_NMDA_receptor       | length=  938 | predicted TM domains=4
GABRA1_GABA-A_receptor    | length=  456 | predicted TM domains=3
SCN1A_sodium_channel      | length= 2009 | predicted TM domains=16
GRIA1_AMPA_receptor       | length=  906 | predicted TM domains=4


## Validation — compare predictions against UniProt's own curated annotations

This is the ground-truth check: how much of your predicted transmembrane regions actually overlap with UniProt's manually curated annotations.

In [ ]:
def fetch_uniprot_annotated_tm_regions(accession: str) -> list:
    """
    Pulls UniProt's manually curated 'Transmembrane' feature annotations
    for a protein via the JSON API. Returns a list of (start, end) tuples
    using the same 0-indexed convention as flag_tm_regions().
    """
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    annotated = []
    for feature in data.get("features", []):
        if feature.get("type") == "Transmembrane":
            loc = feature["location"]
            start = loc["start"]["value"] - 1  # UniProt is 1-indexed
            end = loc["end"]["value"] - 1
            annotated.append((start, end))
    return annotated


def region_overlap_fraction(pred_regions: list, true_regions: list, seq_len: int) -> float:
    """
    Computes what fraction of UniProt's annotated TM residues were also
    captured by your predicted regions. 1.0 = perfect overlap, 0.0 = none.
    """
    pred_mask = np.zeros(seq_len, dtype=bool)
    true_mask = np.zeros(seq_len, dtype=bool)
    for s, e in pred_regions:
        pred_mask[s:e + 1] = True
    for s, e in true_regions:
        true_mask[s:e + 1] = True

    if true_mask.sum() == 0:
        return float("nan")
    overlap = (pred_mask & true_mask).sum()
    return overlap / true_mask.sum()


def validate_protein(name: str, accession: str, seq: str, pred_regions: list):
    """
    Runs the full validation check for one protein and prints a report.
    """
    true_regions = fetch_uniprot_annotated_tm_regions(accession)
    overlap = region_overlap_fraction(pred_regions, true_regions, len(seq))

    print(f"\n--- Validation: {name} ---")
    print(f"UniProt annotated TM domains : {len(true_regions)} {true_regions}")
    print(f"Your predicted TM domains    : {len(pred_regions)} {pred_regions}")
    print(f"Residue-level overlap        : {overlap:.1%}" if overlap == overlap else "No annotated TM regions found")
    return {"protein": name, "num_true": len(true_regions), "num_pred": len(pred_regions), "overlap": overlap}


def run_validation_report(protein_dict: dict):
    """
    Runs the pipeline + validation for every protein and prints a
    clean side-by-side report at the end.
    """
    validation_results = []

    for name, accession in protein_dict.items():
        seq = fetch_uniprot_sequence(accession)
        hydro = sliding_hydrophobicity(seq)
        tm_regions = flag_tm_regions(hydro)

        result = validate_protein(name, accession, seq, tm_regions)
        validation_results.append(result)

    print("\n" + "=" * 60)
    print("FINAL VALIDATION SUMMARY")
    print("=" * 60)
    print(f"{'Protein':28s} | {'True':>5s} | {'Pred':>5s} | {'Overlap':>8s}")
    print("-" * 60)
    for r in validation_results:
        overlap_str = f"{r['overlap']:.1%}" if r['overlap'] == r['overlap'] else "N/A"
        print(f"{r['protein']:28s} | {r['num_true']:5d} | {r['num_pred']:5d} | {overlap_str:>8s}")

    return validation_results

In [ ]:
validation_results = run_validation_report(TARGET_PROTEINS)


--- Validation: GRIN1_NMDA_receptor ---
UniProt annotated TM domains : 3 [(559, 579), (627, 647), (811, 834)]
Your predicted TM domains    : 4 [(0, 25), (557, 581), (630, 651), (810, 833)]
Residue-level overlap        : 93.9%

--- Validation: GABRA1_GABA-A_receptor ---
UniProt annotated TM domains : 4 [(253, 273), (279, 300), (311, 332), (421, 440)]
Your predicted TM domains    : 3 [(254, 271), (310, 331), (423, 437)]
Residue-level overlap        : 63.5%

--- Validation: SCN1A_sodium_channel ---
UniProt annotated TM domains : 24 [(128, 145), (152, 176), (188, 204), (213, 234), (245, 268), (397, 422), (768, 786), (797, 819), (830, 848), (854, 873), (891, 911), (965, 991), (1218, 1236), (1250, 1275), (1278, 1303), (1313, 1331), (1345, 1368), (1457, 1482), (1541, 1559), (1571, 1592), (1601, 1622), (1636, 1654), (1665, 1687), (1759, 1787)]
Your predicted TM domains    : 16 [(114, 142), (240, 269), (394, 425), (760, 784), (800, 818), (835, 857), (880, 911), (961, 995), (1218, 1235), (1331,